# 06 — Error correction: is there a long-run price relationship?

`05_asymmetry.ipynb` worked entirely in *week-over-week price changes* and found that *retail
responds faster to a crude increase than a decrease*. That model says nothing about price
*levels* — whether retail and crude sit in a stable relationship over the long run, or just
happen to trend in the same direction while wandering independently.

Answering that takes a few steps, each a precondition for the next. Both series have to behave
like random walks first — no fixed point either one reverts to, formally a "unit root" — for the
next question to even make sense: two random walks can still be *cointegrated*, moving together
and staying roughly the same distance apart even as each one wanders unpredictably on its own, the
way two people walking with a short rope between them never drift far even though neither is
walking a predictable path. If that holds, there's a real long-run relationship worth fitting
directly — an error-correction model (ECM) — which helps to answer this notebook's question: when retail drifts away from that relationship, does it snap back faster from above (priced too high relative to crude) than from below? 
That's a different question from `05`'s — not "does retail respond faster to a new shock" but "does retail correct back toward its usual relationship with crude faster in one direction." Sections 1–2 check the preconditions, section 3 fits the model, and sections 4–5 test the two things actually of interest.

Same three links as `05`: crude → wholesale, wholesale → retail, and the direct crude → retail
chain.

In [1]:
import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    """Walk upward from `start` until a directory containing pyproject.toml is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("no pyproject.toml found in any parent directory")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.asymmetry import DEFAULT_HAC_MAXLAGS, build_design_matrix, fit_distributed_lag, test_asymmetry
from src.cointegration import adf_test, engle_granger, build_ecm_design_matrix, fit_ecm, test_adjustment_asymmetry
from scripts.verify_alignment import PROCESSED_DIR, RETAIL_SERIES_ID, UPSTREAM_SERIES_ID, load_series

DAILY_CRUDE_SERIES_ID = "DCOILWTICO"
WHOLESALE_SERIES_ID = "WGASUSGULF"

retail_full = load_series(PROCESSED_DIR / "retail.csv", RETAIL_SERIES_ID, "weekly-mon")
weekly_crude = load_series(PROCESSED_DIR / "crude.csv", UPSTREAM_SERIES_ID, "weekly-fri")
daily_crude = load_series(PROCESSED_DIR / "crude.csv", DAILY_CRUDE_SERIES_ID, "daily").dropna(subset=["value"]).reset_index(drop=True)
wholesale_full = load_series(PROCESSED_DIR / "spot.csv", WHOLESALE_SERIES_ID, "weekly-fri")

# Same one-row trim 04/05 use: daily crude's first usable close is 2010-01-04, so any weekly
# series starting on or before that has no valid daily_pit lookback for its first row. Applied
# to every link, not just the daily_pit one, so all three share one sample.
cutoff = daily_crude["date"].min()
retail = retail_full[retail_full["date"] > cutoff].reset_index(drop=True)
wholesale = wholesale_full[wholesale_full["date"] > cutoff].reset_index(drop=True)

links = [
    ("crude → wholesale", wholesale, daily_crude, 1, "daily_pit"),
    ("wholesale → retail", retail, wholesale, 6, "weekly"),
    ("crude → retail", retail, weekly_crude, 4, "weekly"),
]

## 1. Are crude and retail non-stationary in levels?

`adf_test` (Augmented Dickey–Fuller (ADF) test) checks this directly, per series.
The null hypothesis is "this series has a unit root" (behaves like a random walk); we expect to *fail to reject* it (p > 0.05) for all four series below. 
A clean rejection on any of them would mean that series is already stationary in levels, and cointegration would be the wrong tool for it.

In [2]:
series_to_test = {
    "retail": retail,
    "wholesale": wholesale,
    "weekly crude": weekly_crude,
    "daily crude": daily_crude,
}

adf_results = pd.DataFrame(adf_test(df.set_index("date")["value"], name) for name, df in series_to_test.items())
adf_results.round(4)

,name,stat,p_value,crit_1pct,crit_5pct,crit_10pct,likely_unit_root
0,retail,-2.5595,0.1017,-3.4380,-2.8649,-2.5686,True
1,wholesale,-2.5226,0.1101,-3.4380,-2.8649,-2.5686,True
2,weekly crude,-2.5831,0.0965,-3.4380,-2.8649,-2.5686,True
3,daily crude,-2.6257,0.0878,-3.4319,-2.8622,-2.5671,True


## 2. Cointegration test per link

Each link is tested with `engle_granger`: the null hypothesis is "these two series are *not*
cointegrated" (no stable long-run relationship). It also returns `gamma1`, the long-run
pass-through rate implied by the relationship, wherever one exists.

In [3]:
eg_by_link = {}
coint_rows = []
for name, downstream, upstream, k, mode in links:
    eg = engle_granger(downstream, upstream, mode=mode)
    eg_by_link[name] = eg
    coint_rows.append(
        {
            "link": name,
            "K": k,
            "mode": mode,
            "coint_stat": eg["coint_stat"],
            "coint_pvalue": eg["coint_pvalue"],
            "gamma0": eg["gamma0"],
            "gamma1": eg["gamma1"],
        }
    )

coint_results = pd.DataFrame(coint_rows)
coint_results["cointegrated"] = coint_results["coint_pvalue"] < 0.05
coint_results.round(4)

,link,K,mode,coint_stat,coint_pvalue,gamma0,gamma1,cointegrated
0,crude → wholesale,1,daily_pit,-4.2921,0.0026,0.1319,1.1673,True
1,wholesale → retail,6,weekly,-2.8062,0.1637,0.9939,0.9567,False
2,crude → retail,4,weekly,-3.5030,0.0321,1.1447,1.1025,True


Two of the three links cointegrate at the 5% level: crude → wholesale (p = 0.0026) and the
direct crude → retail chain (p = 0.0321). Wholesale → retail does not (p = 0.164) — no
statistical evidence of a stable long-run relationship between wholesale and retail prices in
this sample, even though `05` found the *short-run* asymmetric response concentrated almost
entirely in this same link. The ECM below is fit only for the two links that cointegrate;
wholesale → retail is skipped rather than forced onto a relationship the data doesn't support.

## 3. ECM fit for the cointegrated links

`build_ecm_design_matrix` adds the lagged equilibrium error to `05`'s short-run design matrix —
split into `u_pos_lag1`/`u_neg_lag1`, the same positive/negative decomposition already used for
`Δupstream` there — and `fit_ecm` runs the same HAC-OLS fit on top of it.

In [4]:
res_ecm_by_link = {}
for name, downstream, upstream, k, mode in links:
    eg = eg_by_link[name]
    if eg["coint_pvalue"] >= 0.05:
        print(f"--- {name}: not cointegrated, ECM skipped ---\n")
        continue
    design = build_ecm_design_matrix(downstream, upstream, K=k, gamma0=eg["gamma0"], gamma1=eg["gamma1"], mode=mode)
    res_ecm = fit_ecm(design)
    res_ecm_by_link[name] = res_ecm
    print(f"--- {name} ---")
    print(res_ecm.summary())
    print()

build_design_matrix: dropped 2 of 865 rows to NaN (differencing + 1 lag(s)); 863 rows remain
--- crude → wholesale ---
                            OLS Regression Results                            
Dep. Variable:               d_retail   R-squared:                       0.443
Model:                            OLS   Adj. R-squared:                  0.439
Method:                 Least Squares   F-statistic:                     75.90
Date:                Mon, 17 Aug 2026   Prob (F-statistic):           5.77e-76
Time:                        14:59:09   Log-Likelihood:                 1009.8
No. Observations:                 863   AIC:                            -2006.
Df Residuals:                     856   BIC:                            -1972.
Df Model:                           6                                         
Covariance Type:                  HAC                                         
                  coef    std err          z      P>|z|      [0.025      0.975]
-----------

Fit quality is close to `05`'s plain models — R² = 0.44 for crude → wholesale, 0.51 for
crude → retail — so adding the equilibrium-error term doesn't change how well the short-run part
fits; it adds two more coefficients on top of the same structure.

Crude → wholesale: the same-week response is nearly identical either direction
(`d_up_lag0` = 0.770 vs `d_down_lag0` = 0.802), matching `05`'s finding of no short-run
asymmetry here. `u_pos_lag1`/`u_neg_lag1` are both negative and individually border on
significant (p = 0.048, p = 0.055) — some correction toward the long-run relationship is
happening — but whether the two speeds *differ from each other* is a separate question, tested
next in section 4.

Crude → retail: `d_up_lag0` (0.708) vs `d_down_lag0` (0.256) reproduces `05`'s short-run gap
almost exactly. `u_pos_lag1`/`u_neg_lag1` are smaller here (-0.021, -0.006) and neither is
individually significant (p = 0.25, p = 0.59) — weaker evidence of any correction at all in this
link, before even asking about direction.

## 4. Speed-of-adjustment asymmetry (λ⁺ vs λ⁻)

λ⁺ is the coefficient on `u_pos_lag1` (retail priced above the long-run line the prior week),
λ⁻ on `u_neg_lag1` (priced below). Both should be negative — a positive gap should shrink back
toward zero, a negative gap should grow back toward zero; a positive λ points to a misspecified
model rather than an interesting finding. `test_adjustment_asymmetry` tests whether the two
differ, via `t_test` on a restriction vector — the same approach `05`'s β⁺ vs β⁻ test uses, for
the same reason: the two coefficients are correlated, so their standard errors can't just be
subtracted.

In [5]:
lambda_rows = []
for name, res_ecm in res_ecm_by_link.items():
    adj = test_adjustment_asymmetry(res_ecm)
    lambda_rows.append(
        {
            "link": name,
            "lambda_pos": res_ecm.params["u_pos_lag1"],
            "lambda_neg": res_ecm.params["u_neg_lag1"],
            "diff (lambda+ minus lambda-)": adj["estimate"],
            "se": adj["se"],
            "ci_lo": adj["ci_lo"],
            "ci_hi": adj["ci_hi"],
            "p_value": adj["p_value"],
        }
    )

lambda_results = pd.DataFrame(lambda_rows)
lambda_results.round(4)

,link,lambda_pos,lambda_neg,diff (lambda+ minus lambda-),se,ci_lo,ci_hi,p_value
0,crude → wholesale,-0.0702,-0.0399,-0.0303,0.0482,-0.1248,0.0641,0.5291
1,crude → retail,-0.0208,-0.0064,-0.0144,0.0256,-0.0646,0.0358,0.5740


Both λ⁺ and λ⁻ are negative in both links — the equilibrium error does correct back toward
zero, consistent with genuine error correction rather than a misspecified model. In both links
|λ⁺| > |λ⁻| (correction from above is nominally faster), but the CI on the difference includes
zero in both cases (crude → wholesale: p = 0.53; crude → retail: p = 0.57) — the long-run
adjustment speed is not statistically distinguishable between the two directions in this
sample.

## 5. Short-run asymmetry, re-checked inside the ECM

`05`'s β⁺ vs β⁻ test is re-run on each ECM fit (`res_ecm`), alongside the original `05` fit (no
equilibrium-error term) for direct comparison, at h = 0, 1, and the link's own K.

In [6]:
comparison_rows = []
for name, downstream, upstream, k, mode in links:
    if name not in res_ecm_by_link:
        continue
    res_plain = fit_distributed_lag(build_design_matrix(downstream, upstream, K=k, mode=mode), maxlags=DEFAULT_HAC_MAXLAGS)
    res_ecm = res_ecm_by_link[name]
    for h in sorted({0, 1, k}):
        plain = test_asymmetry(res_plain, K=k, horizon=h)
        ecm = test_asymmetry(res_ecm, K=k, horizon=h)
        comparison_rows.append(
            {
                "link": name,
                "horizon": h,
                "plain_estimate": plain["estimate"],
                "plain_p": plain["p_value"],
                "ecm_estimate": ecm["estimate"],
                "ecm_p": ecm["p_value"],
            }
        )

pd.DataFrame(comparison_rows).round(4)

build_design_matrix: dropped 2 of 865 rows to NaN (differencing + 1 lag(s)); 863 rows remain
build_design_matrix: dropped 5 of 865 rows to NaN (differencing + 4 lag(s)); 860 rows remain


,link,horizon,plain_estimate,plain_p,ecm_estimate,ecm_p
0,crude → wholesale,0,-0.0908,0.4382,-0.0316,0.7921
1,crude → wholesale,1,0.0304,0.8544,0.1410,0.4324
2,crude → retail,0,0.4333,0.0001,0.4515,0.0000
3,crude → retail,1,0.2331,0.0239,0.2777,0.0110
4,crude → retail,4,0.0303,0.8137,0.1288,0.3075


For crude → retail, the short-run asymmetry from `05` survives essentially unchanged: h = 0
(0.433 → 0.452, both p < 0.001) and h = 1 (0.233 → 0.278, both p < 0.03) stay significant with
the equilibrium-error term in the model; h = 4 stays non-significant either way. For crude →
wholesale, `05` found no short-run asymmetry, and that holds inside the ECM too — every estimate's
CI includes zero, in both the plain and ECM versions.

Put together, sections 4 and 5 point in the same direction from two different mechanisms, with
very different statistical strength. A CI that includes zero on λ⁺ − λ⁻ doesn't mean nothing is
happening — section 3 already showed both λ's are negative, and for crude → wholesale
individually near the edge of significance — it means this sample can't tell the two correction
*speeds* apart with confidence. That's a real result, not a failed test: it says the long-run
mechanism, to the extent it's asymmetric at all, is a weaker and noisier effect than the
short-run one.

The short-run asymmetry is the opposite case: `05` already established it, and section 5 shows it
holds up, largely unchanged, once the long-run correction term is added. So for this project, the
asymmetry finding rests on the short-run response — how fast a price move passes through — not on
long-run adjustment speed. That's a narrower claim than "prices correct asymmetrically in every
sense," but it's the one the data actually supports, and it's still the core rockets-and-feathers
result the project set out to test.

## 6. Does the long-run story agree with the short-run story?

Not cleanly — and the two links that cointegrate aren't the link that carries the short-run
effect, so the comparison is smaller than it looks. Wholesale → retail is where `05` found the
asymmetric response, but it isn't cointegrated in this sample, so there's no long-run
relationship to test at all in the link that matters most; the question simply doesn't apply
there.

In the two links that are cointegrated, the two stories partly agree and partly don't. Direction
agrees: λ⁺ is larger in magnitude than λ⁻ in both, meaning that whenever retail sits above its
long-run line with crude, it corrects back somewhat faster than when it sits below — the same
"rockets" shape as the short-run result, restated in levels. But statistical strength does not
agree: the short-run gap (section 5) is significant and survives the ECM at h = 0 and h = 1 for
crude → retail, while the long-run λ⁺ − λ⁻ gap (section 4) is not significant in either
cointegrated link.

Overall, the short-run finding is the one this project can stand behind. The long-run adjustment
story points the same direction but isn't strong enough in this sample to count as a second,
independent confirmation of rockets-and-feathers — and for the link where the effect is
strongest, the long-run question isn't even well-posed.

This doesn't undermine 05's headline result. The short-run asymmetry — the actual rockets-and-feathers claim — has been checked several ways now (extreme weeks, alignment method, lag length, and now: whether it survives adding a long-run correction term) and holds up every time. What this notebook adds is a boundary on the claim: the asymmetry is a short-run repricing-speed effect, not a demonstrated long-run equilibrium-correction effect, and in the link where it's strongest, the long-run question isn't even well-posed.